##1. **Dataset** Setup Explorartion

In [0]:

data_folder = "/Volumes/workspace/default/assignment1_data"

files = [
    "results.csv",
    "goalscorers.csv",
    "shootouts.csv",
    "former_names.csv"
]

for file_name in files:
    file_path = f"{data_folder}/{file_name}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )

    print(f"File: {file_name}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")

    display(df.limit(5))


File: results.csv
Rows: 49945
Columns: 9


date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,FALSE
1873-03-08,England,Scotland,4,2,Friendly,London,England,FALSE
1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,FALSE
1875-03-06,England,Scotland,2,2,Friendly,London,England,FALSE
1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,FALSE


File: goalscorers.csv
Rows: 47914
Columns: 8


date,home_team,away_team,team,scorer,minute,own_goal,penalty
1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44,FALSE,FALSE
1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55,FALSE,FALSE
1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70,FALSE,FALSE
1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75,FALSE,FALSE
1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2,FALSE,FALSE


File: shootouts.csv
Rows: 700
Columns: 5


date,home_team,away_team,winner,first_shooter
1967-08-22,India,Taiwan,Taiwan,null
1970-06-26,Hong Kong,Malaysia,Malaysia,null
1970-06-26,Indonesia,South Korea,South Korea,null
1971-11-14,South Korea,Vietnam Republic,South Korea,null
1972-05-07,South Korea,Iraq,Iraq,null


File: former_names.csv
Rows: 36
Columns: 4


current,former,start_date,end_date
Benin,Dahomey,1959-11-08,1975-11-30
Burkina Faso,Upper Volta,1960-04-14,1984-08-04
Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
Czechoslovakia,Bohemia,1903-04-05,1919-01-01
Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


##2. Dataset docuentation and questions 

### **International Football Results**

## 1. Dataset Overview

**Dataset Title:** International Football Results (1872–2026)

**Source:** International football results dataset maintained by martj42

**Dataset URL:** https://github.com/martj42/international_results

**Access Date:** 2026-10-10

**Format:** CSV

**Number of Source Files:** 4

| File | Rows | Columns |
|---|---:|---:|
| results.csv | 49,945 | 9 |
| goalscorers.csv | 47,914 | 8 |
| shootouts.csv | 700 | 5 |
| former_names.csv | 36 | 4 |

### Why This Dataset Is Interesting

This dataset contains historical international men's football results dating back to 1872. It provides information about match results, national teams, tournaments, goal scorers, penalty shootouts, and historical team names.

The dataset supports analysis across several dimensions, including time, teams, tournaments, and locations. It is suitable for building a data engineering pipeline using Bronze, Silver, and Gold layers.

## 2. Source Grain

- **results.csv:** One row represents one international football match.
- **goalscorers.csv:** One row represents one recorded goal event.
- **shootouts.csv:** One row represents one penalty shootout.
- **former_names.csv:** One row represents a historical team-name relationship and its validity period.

## 3. Candidate Keys

- **results.csv:** A candidate composite key is date, home_team, away_team, home_score, and away_score. This combination is unique in the current snapshot, but may change if a score is corrected.
- **goalscorers.csv:** No reliable natural unique key exists. A generated goal-event identifier will be needed.
- **shootouts.csv:** The combination of date, home_team, and away_team is unique in the current snapshot.
- **former_names.csv:** The combination of current, former, start_date, and end_date is unique in the current snapshot.

## 4. Data Dictionary

| Column | Source | Expected Type | Meaning / Unit |
|---|---|---|---|
| date | results, goalscorers, shootouts | DATE | Date of match |
| home_team | results | STRING | Designated home national team |
| away_team | results | STRING | Designated away national team |
| home_score | results | INTEGER | Home goals, including extra time but not shootouts |
| away_score | results | INTEGER | Away goals, including extra time but not shootouts |
| tournament | results | STRING | Tournament or competition |
| country | results | STRING | Country hosting the match |
| neutral | results | BOOLEAN | Whether match was at a neutral venue |
| team | goalscorers | STRING | Team associated with the goal |
| scorer | goalscorers | STRING | Player credited with scoring |
| minute | goalscorers | INTEGER | Recorded match minute |
| own_goal | goalscorers | BOOLEAN | Own-goal indicator |
| penalty | goalscorers | BOOLEAN | Penalty-goal indicator |
| winner | shootouts | STRING | Penalty shootout winner |
| first_shooter | shootouts | STRING | Team shooting first |
| current | former_names | STRING | Current or successor team name |
| former | former_names | STRING | Historical team name |
| start_date | former_names | DATE | Beginning of historical name period |
| end_date | former_names | DATE | End of historical name period |


### 5. Four Analytical Questions

**Q1:** How has the average number of goals per international match changed across decades?

**Q2:** How does the percentage of home-team victories compare between neutral and non-neutral venues?

**Q3:** Which 10 non-friendly tournaments have hosted the largest number of matches since 2000?

**Q4:** Which 10 national teams have recorded the most match victories since 2000, based on final match scores?

These questions explore time trends, match outcomes, tournament activity, and national team performance.

###Ingestion Strategy

### 1. Full vs. Incremental Loading

The pipeline uses an initial historical load followed by incremental loading. The dataset was divided into three simulated arrival batches:

- **2026-10-07:** Matches before 2000.
- **2026-10-08:** Matches from 2000 to 2014.
- **2026-10-09:** Matches from 2015 onward.

The `run_date` parameter identifies the batch being processed and acts as a simulated ingestion watermark. It is not a true source-update timestamp. If the original dataset were updated regularly, a source modification timestamp would be preferable for detecting newly added or corrected records.

### 2. Batch vs. Streaming

Batch processing was selected because the source consists of historical CSV files rather than continuously arriving events. Real-time streaming is unnecessary for this dataset.

### 3. Push vs. Pull

A pull-based approach was selected because the pipeline initiates reading the CSV files from the Databricks landing zone. The source does not automatically push new data into the Lakehouse.

### 4. Ingestion Frequency

On-demand ingestion was selected for the assignment. In a production environment, the pipeline could run weekly or after new source data becomes available.

### 5. Recovery and Repeatability

Each batch is processed separately through Bronze, Silver, and Gold. Bronze and Silver use batch-specific replacement, while Gold uses SQL MERGE operations to update existing records or insert new ones.

Rerunning the same batch is designed not to create duplicate records. The pipeline also records ingestion counts, validates data quality, and quarantines invalid records.

The three historical batches demonstrate incremental processing without repeatedly loading the entire dataset.

##3. Lakehouse setup and batch preparation 

In [0]:
catalog_name = spark.sql(
    "SELECT current_catalog()"
).first()[0]

print("Catalog:", catalog_name)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS quarantine")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS bronze.landing_zone
""")

landing_path = f"/Volumes/{catalog_name}/bronze/landing_zone"

print("Lakehouse setup completed")
print("Landing zone:", landing_path)

Catalog: workspace
Lakehouse setup completed
Landing zone: /Volumes/workspace/bronze/landing_zone


In [0]:
from pyspark.sql.functions import col

batch_dates = [
    "2026-10-07",
    "2026-10-08",
    "2026-10-09"
]

def save_csv(df, folder, file_name):
    temp_path = f"{folder}/temp_{file_name}"
    final_path = f"{folder}/{file_name}"

    (df.coalesce(1)
       .write
       .mode("overwrite")
       .option("header", "true")
       .csv(temp_path))

    files = dbutils.fs.ls(temp_path)

    for file in files:
        if file.name.startswith("part-") and file.name.endswith(".csv"):
            dbutils.fs.rm(final_path, True)
            dbutils.fs.mv(file.path, final_path)
            break

    dbutils.fs.rm(temp_path, True)


for load_date in batch_dates:
    dbutils.fs.mkdirs(f"{landing_path}/{load_date}")


for file_name in ["results.csv", "goalscorers.csv", "shootouts.csv"]:

    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "false")
          .csv(f"{data_folder}/{file_name}"))

    batches = [
        df.filter(col("date") < "2000-01-01"),
        df.filter((col("date") >= "2000-01-01") &
                  (col("date") < "2015-01-01")),
        df.filter(col("date") >= "2015-01-01")
    ]

    for load_date, batch_df in zip(batch_dates, batches):
        folder = f"{landing_path}/{load_date}"

        save_csv(batch_df, folder, file_name)

        print(load_date, file_name, batch_df.count())


# Historical team names are loaded in the initial batch
former_names = (spark.read
                .option("header", "true")
                .option("inferSchema", "false")
                .csv(f"{data_folder}/former_names.csv"))

save_csv(
    former_names,
    f"{landing_path}/{batch_dates[0]}",
    "former_names.csv"
)

print(batch_dates[0], "former_names.csv", former_names.count())

print("Batch preparation completed")

2026-10-07 results.csv 24066
2026-10-08 results.csv 14366
2026-10-09 results.csv 11513
2026-10-07 goalscorers.csv 19202
2026-10-08 goalscorers.csv 14830
2026-10-09 goalscorers.csv 13882
2026-10-07 shootouts.csv 268
2026-10-08 shootouts.csv 203
2026-10-09 shootouts.csv 229
2026-10-07 former_names.csv 36
Batch preparation completed


##4. Bronze Layer

### Create Bronze tables

In [0]:
%sql
--Create Bronze Layer Tables

USE SCHEMA bronze;

-- Results table
CREATE TABLE IF NOT EXISTS bronze.results (
    date STRING,
    home_team STRING,
    away_team STRING,
    home_score STRING,
    away_score STRING,
    tournament STRING,
    city STRING,
    country STRING,
    neutral STRING,
    _load_date DATE,
    _ingested_at TIMESTAMP,
    _source_file STRING,
    _load_id STRING
)
USING DELTA;


-- Goalscorers table
CREATE TABLE IF NOT EXISTS bronze.goalscorers (
    date STRING,
    home_team STRING,
    away_team STRING,
    team STRING,
    scorer STRING,
    minute STRING,
    own_goal STRING,
    penalty STRING,
    _load_date DATE,
    _ingested_at TIMESTAMP,
    _source_file STRING,
    _load_id STRING
)
USING DELTA;


-- Shootouts table
CREATE TABLE IF NOT EXISTS bronze.shootouts (
    date STRING,
    home_team STRING,
    away_team STRING,
    winner STRING,
    first_shooter STRING,
    _load_date DATE,
    _ingested_at TIMESTAMP,
    _source_file STRING,
    _load_id STRING
)
USING DELTA;


-- Former Names table
CREATE TABLE IF NOT EXISTS bronze.former_names (
    `current` STRING,
    former STRING,
    start_date STRING,
    end_date STRING,
    _load_date DATE,
    _ingested_at TIMESTAMP,
    _source_file STRING,
    _load_id STRING
)
USING DELTA;


-- Batch Audit table
CREATE TABLE IF NOT EXISTS bronze.load_audit (
    load_date DATE,
    source_name STRING,
    load_id STRING,
    input_rows BIGINT,
    bronze_rows BIGINT,
    ingested_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
--  Verification

SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,former_names,false
bronze,goalscorers,false
bronze,load_audit,false
bronze,results,false
bronze,shootouts,false


### Bronze Batch Ingestion

In [0]:
%sql
-- Configuring batch parameters

CREATE WIDGET TEXT run_date DEFAULT "2026-10-07";

CREATE WIDGET TEXT volume_path
DEFAULT "/Volumes/workspace/bronze/landing_zone";

In [0]:
%sql
-- Checking batch parameters

SELECT
    :run_date AS load_date,
    :volume_path AS landing_zone;

load_date,landing_zone
2026-10-09,/Volumes/workspace/bronze/landing_zone


In [0]:
%sql
-- Ingestation of Results

USE SCHEMA bronze;

DELETE FROM bronze.results
WHERE _load_date = to_date(:run_date);

INSERT INTO bronze.results
SELECT
    date,
    home_team,
    away_team,
    home_score,
    away_score,
    tournament,
    city,
    country,
    neutral,
    to_date(:run_date),
    current_timestamp(),
    concat(:volume_path, '/', :run_date, '/results.csv'),
    :run_date
FROM read_files(
    concat(:volume_path, '/', :run_date, '/results.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
11513,11513


In [0]:
%sql
-- Ingesting Goalscorers

DELETE FROM bronze.goalscorers
WHERE _load_date = to_date(:run_date);

INSERT INTO bronze.goalscorers
SELECT
    date,
    home_team,
    away_team,
    team,
    scorer,
    minute,
    own_goal,
    penalty,
    to_date(:run_date),
    current_timestamp(),
    concat(:volume_path, '/', :run_date, '/goalscorers.csv'),
    :run_date
FROM read_files(
    concat(:volume_path, '/', :run_date, '/goalscorers.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
13882,13882


In [0]:
%sql
-- Ingesting Shootouts

DELETE FROM bronze.shootouts
WHERE _load_date = to_date(:run_date);

INSERT INTO bronze.shootouts
SELECT
    date,
    home_team,
    away_team,
    winner,
    first_shooter,
    to_date(:run_date),
    current_timestamp(),
    concat(:volume_path, '/', :run_date, '/shootouts.csv'),
    :run_date
FROM read_files(
    concat(:volume_path, '/', :run_date, '/shootouts.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
229,229


In [0]:
%sql
-- Ingesting of Former Names

DELETE FROM bronze.former_names
WHERE _load_date = to_date(:run_date);

INSERT INTO bronze.former_names
SELECT
    `current`,
    former,
    start_date,
    end_date,
    to_date(:run_date),
    current_timestamp(),
    concat(:volume_path, '/', :run_date, '/former_names.csv'),
    :run_date
FROM read_files(
    concat(:volume_path, '/', :run_date, '/former_names.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
36,36


In [0]:
%sql
-- Record ingestion counts

DELETE FROM bronze.load_audit
WHERE load_date = to_date(:run_date);

INSERT INTO bronze.load_audit
SELECT
    to_date(:run_date),
    'results.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*) FROM bronze.results
     WHERE _load_date = to_date(:run_date)),
    current_timestamp()
FROM read_files(
    concat(:volume_path, '/', :run_date, '/results.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

INSERT INTO bronze.load_audit
SELECT
    to_date(:run_date),
    'goalscorers.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*) FROM bronze.goalscorers
     WHERE _load_date = to_date(:run_date)),
    current_timestamp()
FROM read_files(
    concat(:volume_path, '/', :run_date, '/goalscorers.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

INSERT INTO bronze.load_audit
SELECT
    to_date(:run_date),
    'shootouts.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*) FROM bronze.shootouts
     WHERE _load_date = to_date(:run_date)),
    current_timestamp()
FROM read_files(
    concat(:volume_path, '/', :run_date, '/shootouts.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

INSERT INTO bronze.load_audit
SELECT
    to_date(:run_date),
    'former_names.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*) FROM bronze.former_names
     WHERE _load_date = to_date(:run_date)),
    current_timestamp()
FROM read_files(
    concat(:volume_path, '/', :run_date, '/former_names.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- Verify Bronze ingestion

SELECT
    load_date,
    source_name,
    input_rows,
    bronze_rows
FROM bronze.load_audit
WHERE load_date = to_date(:run_date)
ORDER BY source_name;

load_date,source_name,input_rows,bronze_rows
2026-10-07,former_names.csv,36,36
2026-10-07,goalscorers.csv,19202,19202
2026-10-07,results.csv,24066,24066
2026-10-07,shootouts.csv,268,268


## 5. Silver Layer & Quarantine

### Data Quality Exploration

In [0]:
%sql
-- Check Results Data Quality

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        TRY_CAST(date AS DATE) IS NULL
    ) AS invalid_dates,

    COUNT_IF(
        home_team IS NULL OR TRIM(home_team) = ''
        OR away_team IS NULL OR TRIM(away_team) = ''
    ) AS missing_teams,

    COUNT_IF(
        TRY_CAST(home_score AS INT) IS NULL
        OR TRY_CAST(away_score AS INT) IS NULL
        OR TRY_CAST(home_score AS INT) < 0
        OR TRY_CAST(away_score AS INT) < 0
    ) AS invalid_scores

FROM bronze.results
WHERE _load_date = TO_DATE(:run_date);

total_rows,invalid_dates,missing_teams,invalid_scores
24066,0,0,0


In [0]:
%sql
-- Check Goalscorers Data Quality

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        TRY_CAST(date AS DATE) IS NULL
    ) AS invalid_dates,

    COUNT_IF(
        TRY_CAST(minute AS INT) IS NULL
    ) AS unknown_minutes,

    COUNT_IF(
        scorer IS NULL OR TRIM(scorer) = ''
        OR team IS NULL OR TRIM(team) = ''
    ) AS missing_information

FROM bronze.goalscorers
WHERE _load_date = TO_DATE(:run_date);

total_rows,invalid_dates,unknown_minutes,missing_information
19202,0,254,0


In [0]:
%sql
-- Check Shootouts Data Quality

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        TRY_CAST(date AS DATE) IS NULL
    ) AS invalid_dates,

    COUNT_IF(
        winner IS NULL OR TRIM(winner) = ''
    ) AS missing_winners,

    COUNT_IF(
        first_shooter IS NULL
        OR TRIM(first_shooter) = ''
    ) AS missing_first_shooter

FROM bronze.shootouts
WHERE _load_date = TO_DATE(:run_date);

total_rows,invalid_dates,missing_winners,missing_first_shooter
268,0,0,212


In [0]:
%sql
-- Check Potential Duplicate Matches

SELECT
    date,
    home_team,
    away_team,
    COUNT(*) AS match_count

FROM bronze.results

WHERE _load_date = TO_DATE(:run_date)

GROUP BY
    date,
    home_team,
    away_team

HAVING COUNT(*) > 1;

date,home_team,away_team,match_count
1974-02-17,Tahiti,New Caledonia,2



### Silver Layer — Initial Data Quality Exploration

Before performing Silver transformations, the Bronze tables were examined for potential data quality issues.

The following checks were performed:

1. Invalid match dates.
2. Missing home and away teams.
3. Invalid or negative match scores.
4. Unknown goal-scoring minutes.
5. Missing penalty shootout information.
6. Potential duplicate match identifiers.

The initial batch contains unknown goal minutes and missing first-shooter information. These values should be handled appropriately rather than automatically removing the records.

Potential duplicate matches also require careful investigation because two matches can share the same date and teams.

These findings will guide the cleaning, validation, and quarantine rules in the Silver layer.

### Silver & Quarantine Table creation

In [0]:
%sql
-- Create Silver Layer Tables

-- Cleaned Match Results
CREATE TABLE IF NOT EXISTS silver.results (
    match_id STRING,
    date DATE,
    home_team STRING,
    away_team STRING,
    home_score INT,
    away_score INT,
    tournament STRING,
    city STRING,
    country STRING,
    neutral BOOLEAN,
    _load_date DATE
)
USING DELTA;


--  Cleaned Goalscorers
CREATE TABLE IF NOT EXISTS silver.goalscorers (
    goal_id STRING,
    match_id STRING,
    date DATE,
    home_team STRING,
    away_team STRING,
    team STRING,
    scorer STRING,
    minute INT,
    own_goal BOOLEAN,
    penalty BOOLEAN,
    _load_date DATE
)
USING DELTA;


-- Cleaned Shootouts
CREATE TABLE IF NOT EXISTS silver.shootouts (
    match_id STRING,
    date DATE,
    home_team STRING,
    away_team STRING,
    winner STRING,
    first_shooter STRING,
    _load_date DATE
)
USING DELTA;


-- Cleaned Former Names
CREATE TABLE IF NOT EXISTS silver.former_names (
    `current` STRING,
    former STRING,
    start_date DATE,
    end_date DATE,
    _load_date DATE
)
USING DELTA;

In [0]:
%sql
-- Create Quarantine Tables

-- 1) Invalid Match Results
CREATE TABLE IF NOT EXISTS quarantine.results (
    date STRING,
    home_team STRING,
    away_team STRING,
    home_score STRING,
    away_score STRING,
    tournament STRING,
    city STRING,
    country STRING,
    neutral STRING,
    rejection_reason STRING,
    _load_date DATE
)
USING DELTA;


-- 2) Invalid Goalscorers
CREATE TABLE IF NOT EXISTS quarantine.goalscorers (
    date STRING,
    home_team STRING,
    away_team STRING,
    team STRING,
    scorer STRING,
    minute STRING,
    own_goal STRING,
    penalty STRING,
    rejection_reason STRING,
    _load_date DATE
)
USING DELTA;


-- 3) Invalid Shootouts
CREATE TABLE IF NOT EXISTS quarantine.shootouts (
    date STRING,
    home_team STRING,
    away_team STRING,
    winner STRING,
    first_shooter STRING,
    rejection_reason STRING,
    _load_date DATE
)
USING DELTA;


-- 4)Invalid Former Names
CREATE TABLE IF NOT EXISTS quarantine.former_names (
    `current` STRING,
    former STRING,
    start_date STRING,
    end_date STRING,
    rejection_reason STRING,
    _load_date DATE
)
USING DELTA;

In [0]:
%sql
-- Verify Silver and Quarantine Tables

SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema IN ('silver', 'quarantine')
ORDER BY table_schema, table_name;

table_schema,table_name
quarantine,former_names
quarantine,goalscorers
quarantine,results
quarantine,shootouts
silver,former_names
silver,goalscorers
silver,results
silver,shootouts


### Silver and Quarantine Layer Design

The Silver layer stores cleaned and validated football data.

Four Silver Delta tables were created to store match results, goal-scoring records, penalty shootouts, and historical team names.

Unlike Bronze, Silver uses appropriate data types such as DATE, INT, and BOOLEAN.

Four Quarantine tables were also created to isolate invalid records. Each Quarantine table preserves the original values and includes a rejection reason and load date.

Unknown goal minutes and missing first-shooter information will not automatically cause rejection because these fields may legitimately be unavailable.

This design follows the instructor's example and supports incremental batch processing.

### Cleaning & Validation

In [0]:
%sql
-- Set current batch date

DECLARE OR REPLACE VARIABLE batch_date DATE
DEFAULT DATE '2026-10-07';

SET VAR batch_date = TO_DATE(:run_date);

SELECT session.batch_date AS current_batch;

current_batch
2026-10-09


In [0]:
%sql
-- Clean match results

CREATE OR REPLACE TEMPORARY VIEW stg_results_cleaned AS

WITH cleaned AS (
    SELECT
        TRY_CAST(date AS DATE) AS date,
        NULLIF(TRIM(home_team), '') AS home_team,
        NULLIF(TRIM(away_team), '') AS away_team,
        TRY_CAST(home_score AS INT) AS home_score,
        TRY_CAST(away_score AS INT) AS away_score,
        NULLIF(TRIM(tournament), '') AS tournament,
        NULLIF(TRIM(city), '') AS city,
        NULLIF(TRIM(country), '') AS country,
        TRY_CAST(neutral AS BOOLEAN) AS neutral,
        _load_date
    FROM bronze.results
    WHERE _load_date = session.batch_date
)

SELECT
    SHA2(CONCAT_WS('|',
        CAST(date AS STRING),
        home_team,
        away_team,
        CAST(home_score AS STRING),
        CAST(away_score AS STRING),
        tournament,
        city,
        country,
        CAST(neutral AS STRING)
    ), 256) AS match_id,

    *,

    CASE
        WHEN date IS NULL
            THEN 'Invalid date'
        WHEN home_team IS NULL OR away_team IS NULL
            THEN 'Missing team'
        WHEN home_team = away_team
            THEN 'Same home and away team'
        WHEN home_score IS NULL OR away_score IS NULL
            THEN 'Invalid score'
        WHEN home_score < 0 OR away_score < 0
            THEN 'Negative score'
        WHEN tournament IS NULL OR country IS NULL
            THEN 'Missing tournament or country'
        WHEN neutral IS NULL
            THEN 'Invalid neutral value'
        ELSE NULL
    END AS rejection_reason

FROM cleaned;

In [0]:
%sql
-- Prepare match lookup

CREATE OR REPLACE TEMPORARY VIEW stg_match_lookup AS

SELECT
    date,
    home_team,
    away_team,
    MAX(match_id) AS match_id

FROM stg_results_cleaned

WHERE rejection_reason IS NULL

GROUP BY
    date,
    home_team,
    away_team

HAVING COUNT(*) = 1;

In [0]:
%sql
-- Clean goalscorers

CREATE OR REPLACE TEMPORARY VIEW stg_goalscorers_cleaned AS

SELECT
    CONCAT(
        DATE_FORMAT(g._load_date, 'yyyyMMdd'),
        '-',
        CAST(ROW_NUMBER() OVER (
            ORDER BY
                g.date,
                g.home_team,
                g.away_team,
                g.team,
                g.scorer,
                g.minute,
                g.own_goal,
                g.penalty
        ) AS STRING)
    ) AS goal_id,

    m.match_id,

    TRY_CAST(g.date AS DATE) AS date,
    TRIM(g.home_team) AS home_team,
    TRIM(g.away_team) AS away_team,
    TRIM(g.team) AS team,

    CASE
        WHEN UPPER(TRIM(g.scorer)) IN ('NA', '')
            THEN NULL
        ELSE TRIM(g.scorer)
    END AS scorer,

    TRY_CAST(g.minute AS INT) AS minute,
    TRY_CAST(g.own_goal AS BOOLEAN) AS own_goal,
    TRY_CAST(g.penalty AS BOOLEAN) AS penalty,

    g._load_date,

    CASE
        WHEN TRY_CAST(g.date AS DATE) IS NULL
            THEN 'Invalid date'

        WHEN g.team IS NULL OR TRIM(g.team) = ''
            THEN 'Missing team'

        WHEN TRIM(g.team) NOT IN (
            TRIM(g.home_team), TRIM(g.away_team)
        )
            THEN 'Team not in match'

        WHEN TRY_CAST(g.own_goal AS BOOLEAN) IS NULL
          OR TRY_CAST(g.penalty AS BOOLEAN) IS NULL
            THEN 'Invalid goal indicator'

        WHEN UPPER(TRIM(g.minute)) NOT IN ('NA', '')
             AND TRY_CAST(g.minute AS INT) IS NULL
            THEN 'Invalid minute'

        WHEN TRY_CAST(g.minute AS INT) < 1
          OR TRY_CAST(g.minute AS INT) > 130
            THEN 'Minute outside expected range'

        WHEN m.match_id IS NULL
            THEN 'No unique matching result'

        ELSE NULL
    END AS rejection_reason

FROM bronze.goalscorers g

LEFT JOIN stg_match_lookup m
    ON TRY_CAST(g.date AS DATE) = m.date
    AND TRIM(g.home_team) = m.home_team
    AND TRIM(g.away_team) = m.away_team

WHERE g._load_date = session.batch_date;

In [0]:
%sql
-- Clean shootouts

CREATE OR REPLACE TEMPORARY VIEW stg_shootouts_cleaned AS

SELECT
    m.match_id,

    TRY_CAST(s.date AS DATE) AS date,
    TRIM(s.home_team) AS home_team,
    TRIM(s.away_team) AS away_team,
    TRIM(s.winner) AS winner,
    NULLIF(TRIM(s.first_shooter), '') AS first_shooter,
    s._load_date,

    CASE
        WHEN TRY_CAST(s.date AS DATE) IS NULL
            THEN 'Invalid date'

        WHEN s.winner IS NULL OR TRIM(s.winner) = ''
            THEN 'Missing winner'

        WHEN TRIM(s.winner) NOT IN (
            TRIM(s.home_team), TRIM(s.away_team)
        )
            THEN 'Winner not in match'

        WHEN NULLIF(TRIM(s.first_shooter), '') IS NOT NULL
          AND TRIM(s.first_shooter) NOT IN (
              TRIM(s.home_team), TRIM(s.away_team)
          )
            THEN 'Invalid first shooter'

        WHEN m.match_id IS NULL
            THEN 'No unique matching result'

        ELSE NULL
    END AS rejection_reason

FROM bronze.shootouts s

LEFT JOIN stg_match_lookup m
    ON TRY_CAST(s.date AS DATE) = m.date
    AND TRIM(s.home_team) = m.home_team
    AND TRIM(s.away_team) = m.away_team

WHERE s._load_date = session.batch_date;

In [0]:
%sql
-- Clean former team names

CREATE OR REPLACE TEMPORARY VIEW stg_former_names_cleaned AS

WITH cleaned AS (
    SELECT
        NULLIF(TRIM(`current`), '') AS `current`,
        NULLIF(TRIM(former), '') AS former,
        TRY_CAST(start_date AS DATE) AS start_date,
        TRY_CAST(end_date AS DATE) AS end_date,
        _load_date
    FROM bronze.former_names
    WHERE _load_date = session.batch_date
)

SELECT
    *,

    CASE
        WHEN `current` IS NULL OR former IS NULL
            THEN 'Missing team name'

        WHEN start_date IS NULL OR end_date IS NULL
            THEN 'Invalid date'

        WHEN start_date > end_date
            THEN 'Start date after end date'

        ELSE NULL
    END AS rejection_reason

FROM cleaned;

In [0]:
%sql
-- Validation summary

SELECT
    'results.csv' AS source_file,
    COUNT(*) AS total_rows,
    COUNT_IF(rejection_reason IS NULL) AS valid_rows,
    COUNT_IF(rejection_reason IS NOT NULL) AS invalid_rows
FROM stg_results_cleaned

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_goalscorers_cleaned

UNION ALL

SELECT
    'shootouts.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_shootouts_cleaned

UNION ALL

SELECT
    'former_names.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_former_names_cleaned;

source_file,total_rows,valid_rows,invalid_rows
results.csv,24066,24066,0
goalscorers.csv,19202,19202,0
shootouts.csv,268,268,0
former_names.csv,36,36,0


In [0]:
%sql
-- Check goal identifiers and missing information

SELECT
    COUNT(*) AS total_goals,
    COUNT(DISTINCT goal_id) AS unique_goal_ids,
    COUNT_IF(minute IS NULL) AS unknown_minutes,
    COUNT_IF(scorer IS NULL) AS unknown_scorers
FROM stg_goalscorers_cleaned;

total_goals,unique_goal_ids,unknown_minutes,unknown_scorers
19202,19202,254,44


In [0]:
%sql
-- Verify temporary staging views

SELECT
    (SELECT COUNT(*) FROM stg_results_cleaned)
        AS results_rows,

    (SELECT COUNT(*) FROM stg_match_lookup)
        AS match_lookup_rows,

    (SELECT COUNT(*) FROM stg_goalscorers_cleaned)
        AS goalscorers_rows,

    (SELECT COUNT(*) FROM stg_shootouts_cleaned)
        AS shootouts_rows,

    (SELECT COUNT(*) FROM stg_former_names_cleaned)
        AS former_names_rows;

results_rows,match_lookup_rows,goalscorers_rows,shootouts_rows,former_names_rows
24066,24064,19202,268,36


### Silver & Quarantine Loading

In [0]:
%sql
--  Prepare Quarantine Records
-- Restore the batch date variable

DECLARE OR REPLACE VARIABLE batch_date DATE
DEFAULT DATE '2026-10-07';

SET VAR batch_date = TO_DATE(:run_date);
-- Invalid match results
CREATE OR REPLACE TEMPORARY VIEW stg_results_rejected AS
SELECT
    r.date,
    r.home_team,
    r.away_team,
    r.home_score,
    r.away_score,
    r.tournament,
    r.city,
    r.country,
    r.neutral,

    CASE
        WHEN TRY_CAST(r.date AS DATE) IS NULL
            THEN 'Invalid date'
        WHEN NULLIF(TRIM(r.home_team), '') IS NULL
          OR NULLIF(TRIM(r.away_team), '') IS NULL
            THEN 'Missing team'
        WHEN TRIM(r.home_team) = TRIM(r.away_team)
            THEN 'Same home and away team'
        WHEN TRY_CAST(r.home_score AS INT) IS NULL
          OR TRY_CAST(r.away_score AS INT) IS NULL
            THEN 'Invalid score'
        WHEN TRY_CAST(r.home_score AS INT) < 0
          OR TRY_CAST(r.away_score AS INT) < 0
            THEN 'Negative score'
        WHEN NULLIF(TRIM(r.tournament), '') IS NULL
          OR NULLIF(TRIM(r.country), '') IS NULL
            THEN 'Missing tournament or country'
        WHEN TRY_CAST(r.neutral AS BOOLEAN) IS NULL
            THEN 'Invalid neutral value'
        ELSE NULL
    END AS rejection_reason,

    r._load_date

FROM bronze.results r
WHERE r._load_date = session.batch_date;


-- Invalid goal records
CREATE OR REPLACE TEMPORARY VIEW stg_goalscorers_rejected AS
SELECT
    g.date,
    g.home_team,
    g.away_team,
    g.team,
    g.scorer,
    g.minute,
    g.own_goal,
    g.penalty,

    CASE
        WHEN TRY_CAST(g.date AS DATE) IS NULL
            THEN 'Invalid date'
        WHEN g.team IS NULL OR TRIM(g.team) = ''
            THEN 'Missing team'
        WHEN TRIM(g.team) NOT IN (
            TRIM(g.home_team), TRIM(g.away_team)
        )
            THEN 'Team not in match'
        WHEN TRY_CAST(g.own_goal AS BOOLEAN) IS NULL
          OR TRY_CAST(g.penalty AS BOOLEAN) IS NULL
            THEN 'Invalid goal indicator'
        WHEN UPPER(TRIM(g.minute)) NOT IN ('NA', '')
          AND TRY_CAST(g.minute AS INT) IS NULL
            THEN 'Invalid minute'
        WHEN TRY_CAST(g.minute AS INT) < 1
          OR TRY_CAST(g.minute AS INT) > 130
            THEN 'Minute outside expected range'
        WHEN m.match_id IS NULL
            THEN 'No unique matching result'
        ELSE NULL
    END AS rejection_reason,

    g._load_date

FROM bronze.goalscorers g

LEFT JOIN stg_match_lookup m
    ON TRY_CAST(g.date AS DATE) = m.date
    AND TRIM(g.home_team) = m.home_team
    AND TRIM(g.away_team) = m.away_team

WHERE g._load_date = session.batch_date;


-- Invalid shootout records
CREATE OR REPLACE TEMPORARY VIEW stg_shootouts_rejected AS
SELECT
    s.date,
    s.home_team,
    s.away_team,
    s.winner,
    s.first_shooter,

    CASE
        WHEN TRY_CAST(s.date AS DATE) IS NULL
            THEN 'Invalid date'
        WHEN s.winner IS NULL OR TRIM(s.winner) = ''
            THEN 'Missing winner'
        WHEN TRIM(s.winner) NOT IN (
            TRIM(s.home_team), TRIM(s.away_team)
        )
            THEN 'Winner not in match'
        WHEN NULLIF(TRIM(s.first_shooter), '') IS NOT NULL
          AND TRIM(s.first_shooter) NOT IN (
              TRIM(s.home_team), TRIM(s.away_team)
          )
            THEN 'Invalid first shooter'
        WHEN m.match_id IS NULL
            THEN 'No unique matching result'
        ELSE NULL
    END AS rejection_reason,

    s._load_date

FROM bronze.shootouts s

LEFT JOIN stg_match_lookup m
    ON TRY_CAST(s.date AS DATE) = m.date
    AND TRIM(s.home_team) = m.home_team
    AND TRIM(s.away_team) = m.away_team

WHERE s._load_date = session.batch_date;


-- Invalid former-name records
CREATE OR REPLACE TEMPORARY VIEW stg_former_names_rejected AS
SELECT
    f.`current`,
    f.former,
    f.start_date,
    f.end_date,

    CASE
        WHEN NULLIF(TRIM(f.`current`), '') IS NULL
          OR NULLIF(TRIM(f.former), '') IS NULL
            THEN 'Missing team name'
        WHEN TRY_CAST(f.start_date AS DATE) IS NULL
          OR TRY_CAST(f.end_date AS DATE) IS NULL
            THEN 'Invalid date'
        WHEN TRY_CAST(f.start_date AS DATE)
           > TRY_CAST(f.end_date AS DATE)
            THEN 'Start date after end date'
        ELSE NULL
    END AS rejection_reason,

    f._load_date

FROM bronze.former_names f
WHERE f._load_date = session.batch_date;

In [0]:
%sql
-- Load Valid Records into Silver

-- Results
DELETE FROM silver.results
WHERE _load_date = session.batch_date;

INSERT INTO silver.results
SELECT DISTINCT
    match_id,
    date,
    home_team,
    away_team,
    home_score,
    away_score,
    tournament,
    city,
    country,
    neutral,
    _load_date
FROM stg_results_cleaned
WHERE rejection_reason IS NULL;


-- Goalscorers
DELETE FROM silver.goalscorers
WHERE _load_date = session.batch_date;

INSERT INTO silver.goalscorers
SELECT
    goal_id,
    match_id,
    date,
    home_team,
    away_team,
    team,
    scorer,
    minute,
    own_goal,
    penalty,
    _load_date
FROM stg_goalscorers_cleaned
WHERE rejection_reason IS NULL;


-- Shootouts
DELETE FROM silver.shootouts
WHERE _load_date = session.batch_date;

INSERT INTO silver.shootouts
SELECT DISTINCT
    match_id,
    date,
    home_team,
    away_team,
    winner,
    first_shooter,
    _load_date
FROM stg_shootouts_cleaned
WHERE rejection_reason IS NULL;


-- Former Names
DELETE FROM silver.former_names
WHERE _load_date = session.batch_date;

INSERT INTO silver.former_names
SELECT DISTINCT
    `current`,
    former,
    start_date,
    end_date,
    _load_date
FROM stg_former_names_cleaned
WHERE rejection_reason IS NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
--  Load Invalid Records into Quarantine

-- Results
DELETE FROM quarantine.results
WHERE _load_date = session.batch_date;

INSERT INTO quarantine.results
SELECT *
FROM stg_results_rejected
WHERE rejection_reason IS NOT NULL;


-- Goalscorers
DELETE FROM quarantine.goalscorers
WHERE _load_date = session.batch_date;

INSERT INTO quarantine.goalscorers
SELECT *
FROM stg_goalscorers_rejected
WHERE rejection_reason IS NOT NULL;


-- Shootouts
DELETE FROM quarantine.shootouts
WHERE _load_date = session.batch_date;

INSERT INTO quarantine.shootouts
SELECT *
FROM stg_shootouts_rejected
WHERE rejection_reason IS NOT NULL;


-- Former Names
DELETE FROM quarantine.former_names
WHERE _load_date = session.batch_date;

INSERT INTO quarantine.former_names
SELECT *
FROM stg_former_names_rejected
WHERE rejection_reason IS NOT NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
-- Verify Silver and Quarantine Counts

SELECT
    'results.csv' AS source_file,
    (SELECT COUNT(*) FROM bronze.results
     WHERE _load_date = session.batch_date) AS bronze_rows,
    (SELECT COUNT(*) FROM silver.results
     WHERE _load_date = session.batch_date) AS silver_rows,
    (SELECT COUNT(*) FROM quarantine.results
     WHERE _load_date = session.batch_date) AS quarantine_rows

UNION ALL

SELECT
    'goalscorers.csv',
    (SELECT COUNT(*) FROM bronze.goalscorers
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM silver.goalscorers
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM quarantine.goalscorers
     WHERE _load_date = session.batch_date)

UNION ALL

SELECT
    'shootouts.csv',
    (SELECT COUNT(*) FROM bronze.shootouts
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM silver.shootouts
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM quarantine.shootouts
     WHERE _load_date = session.batch_date)

UNION ALL

SELECT
    'former_names.csv',
    (SELECT COUNT(*) FROM bronze.former_names
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM silver.former_names
     WHERE _load_date = session.batch_date),
    (SELECT COUNT(*) FROM quarantine.former_names
     WHERE _load_date = session.batch_date);

source_file,bronze_rows,silver_rows,quarantine_rows
results.csv,24066,24066,0
goalscorers.csv,19202,19202,0
shootouts.csv,268,268,0
former_names.csv,36,36,0


### Silver and Quarantine Loading

The validated records from the temporary staging views were loaded into the Silver Delta tables.

Records that failed the validation rules were routed to Quarantine tables with a rejection reason.

The Silver layer stores cleaned values using appropriate data types, while Quarantine preserves rejected source values for investigation.

Duplicate handling was applied to match results, shootouts, and historical team-name mappings. Repeated goal events were preserved because identical-looking rows may represent separate goals.

The loading process is batch-specific and repeatable. Existing records for the selected load date are deleted before that batch is reinserted.

This approach follows the instructor's incremental medallion architecture example.

### Data Quality reporting

In [0]:
%sql
-- Bronze, Silver and Quarantine Counts

WITH row_counts AS (

    SELECT
        'results.csv' AS source_file,
        (SELECT COUNT(*) FROM bronze.results
         WHERE _load_date = TO_DATE(:run_date)) AS bronze_rows,
        (SELECT COUNT(*) FROM silver.results
         WHERE _load_date = TO_DATE(:run_date)) AS silver_rows,
        (SELECT COUNT(*) FROM quarantine.results
         WHERE _load_date = TO_DATE(:run_date)) AS quarantine_rows

    UNION ALL

    SELECT
        'goalscorers.csv',
        (SELECT COUNT(*) FROM bronze.goalscorers
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM silver.goalscorers
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM quarantine.goalscorers
         WHERE _load_date = TO_DATE(:run_date))

    UNION ALL

    SELECT
        'shootouts.csv',
        (SELECT COUNT(*) FROM bronze.shootouts
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM silver.shootouts
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM quarantine.shootouts
         WHERE _load_date = TO_DATE(:run_date))

    UNION ALL

    SELECT
        'former_names.csv',
        (SELECT COUNT(*) FROM bronze.former_names
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM silver.former_names
         WHERE _load_date = TO_DATE(:run_date)),
        (SELECT COUNT(*) FROM quarantine.former_names
         WHERE _load_date = TO_DATE(:run_date))
)

SELECT
    *,
    bronze_rows - silver_rows - quarantine_rows
        AS row_difference
FROM row_counts
ORDER BY source_file;

source_file,bronze_rows,silver_rows,quarantine_rows,row_difference
former_names.csv,36,36,0,0
goalscorers.csv,19202,19202,0,0
results.csv,24066,24066,0,0
shootouts.csv,268,268,0,0


In [0]:
%sql
-- Count Exact Duplicate Source Rows

SELECT
    'results.csv' AS source_file,
    COALESCE(SUM(row_count - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS row_count
    FROM bronze.results
    WHERE _load_date = TO_DATE(:run_date)
    GROUP BY
        date, home_team, away_team,
        home_score, away_score,
        tournament, city, country, neutral
)

UNION ALL

SELECT
    'goalscorers.csv',
    COALESCE(SUM(row_count - 1), 0)
FROM (
    SELECT COUNT(*) AS row_count
    FROM bronze.goalscorers
    WHERE _load_date = TO_DATE(:run_date)
    GROUP BY
        date, home_team, away_team,
        team, scorer, minute, own_goal, penalty
)

UNION ALL

SELECT
    'shootouts.csv',
    COALESCE(SUM(row_count - 1), 0)
FROM (
    SELECT COUNT(*) AS row_count
    FROM bronze.shootouts
    WHERE _load_date = TO_DATE(:run_date)
    GROUP BY
        date, home_team, away_team,
        winner, first_shooter
)

UNION ALL

SELECT
    'former_names.csv',
    COALESCE(SUM(row_count - 1), 0)
FROM (
    SELECT COUNT(*) AS row_count
    FROM bronze.former_names
    WHERE _load_date = TO_DATE(:run_date)
    GROUP BY
        `current`, former, start_date, end_date
);

source_file,duplicate_rows
results.csv,0
goalscorers.csv,70
shootouts.csv,0
former_names.csv,0


In [0]:
%sql
-- Critical Null Checks

SELECT
    'results.csv' AS source_file,
    COUNT_IF(
        match_id IS NULL OR date IS NULL
        OR home_team IS NULL OR away_team IS NULL
        OR home_score IS NULL OR away_score IS NULL
        OR tournament IS NULL OR country IS NULL
        OR neutral IS NULL
    ) AS rows_missing_critical_values
FROM silver.results
WHERE _load_date = TO_DATE(:run_date)

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT_IF(
        goal_id IS NULL OR match_id IS NULL
        OR date IS NULL OR home_team IS NULL
        OR away_team IS NULL OR team IS NULL
        OR own_goal IS NULL OR penalty IS NULL
    )
FROM silver.goalscorers
WHERE _load_date = TO_DATE(:run_date)

UNION ALL

SELECT
    'shootouts.csv',
    COUNT_IF(
        match_id IS NULL OR date IS NULL
        OR home_team IS NULL OR away_team IS NULL
        OR winner IS NULL
    )
FROM silver.shootouts
WHERE _load_date = TO_DATE(:run_date)

UNION ALL

SELECT
    'former_names.csv',
    COUNT_IF(
        `current` IS NULL OR former IS NULL
        OR start_date IS NULL OR end_date IS NULL
    )
FROM silver.former_names
WHERE _load_date = TO_DATE(:run_date);

source_file,rows_missing_critical_values
results.csv,0
goalscorers.csv,0
shootouts.csv,0
former_names.csv,0


In [0]:
%sql
-- Optional Missing Values

SELECT
    (SELECT COUNT(*)
     FROM silver.goalscorers
     WHERE _load_date = TO_DATE(:run_date)
       AND minute IS NULL) AS unknown_minutes,

    (SELECT COUNT(*)
     FROM silver.goalscorers
     WHERE _load_date = TO_DATE(:run_date)
       AND scorer IS NULL) AS unknown_scorers,

    (SELECT COUNT(*)
     FROM silver.shootouts
     WHERE _load_date = TO_DATE(:run_date)
       AND first_shooter IS NULL) AS unknown_first_shooters;

unknown_minutes,unknown_scorers,unknown_first_shooters
254,44,212


In [0]:
%sql
-- Silver Quality Gates

SELECT

    -- Duplicate match identifiers
    (SELECT COUNT(*) - COUNT(DISTINCT match_id)
     FROM silver.results
     WHERE _load_date = TO_DATE(:run_date))
     AS duplicate_match_ids,

    -- Duplicate goal identifiers
    (SELECT COUNT(*) - COUNT(DISTINCT goal_id)
     FROM silver.goalscorers
     WHERE _load_date = TO_DATE(:run_date))
     AS duplicate_goal_ids,

    -- Duplicate shootout identifiers
    (SELECT COUNT(*) - COUNT(DISTINCT match_id)
     FROM silver.shootouts
     WHERE _load_date = TO_DATE(:run_date))
     AS duplicate_shootout_ids,

    -- Goals without a matching result
    (SELECT COUNT(*)
     FROM silver.goalscorers g
     LEFT JOIN silver.results r
         ON g.match_id = r.match_id
     WHERE g._load_date = TO_DATE(:run_date)
       AND r.match_id IS NULL)
     AS goals_without_match,

    -- Shootouts without a matching result
    (SELECT COUNT(*)
     FROM silver.shootouts s
     LEFT JOIN silver.results r
         ON s.match_id = r.match_id
     WHERE s._load_date = TO_DATE(:run_date)
       AND r.match_id IS NULL)
     AS shootouts_without_match,

    -- Invalid match scores
    (SELECT COUNT(*)
     FROM silver.results
     WHERE _load_date = TO_DATE(:run_date)
       AND (
           home_score < 0
           OR away_score < 0
           OR home_team = away_team
       ))
     AS invalid_matches;

duplicate_match_ids,duplicate_goal_ids,duplicate_shootout_ids,goals_without_match,shootouts_without_match,invalid_matches
0,0,0,0,0,0


### Silver Layer — Data Quality Report

A data-quality report was generated to evaluate the first batch of the pipeline.

The report includes:

1. Bronze, Silver, and Quarantine row counts.
2. Detection of exact duplicate source records.
3. Null checks for critical and optional fields.
4. Uniqueness checks for generated identifiers.
5. Referential-integrity checks between related tables.
6. Validation of match scores.

The initial batch contains 70 excess identical-looking goalscorer rows. These records were preserved because the source does not provide enough information to determine whether they represent duplicated records or separate goals with unknown minutes.

Missing goal minutes, scorer names, and first-shooter information were retained as NULL values rather than rejecting otherwise valid records.

The quality checks help ensure that Silver contains trusted data while documenting known limitations.

These checks will be repeated when additional batches are processed.

## 6. Gold Layer — Star Schema


The Gold layer uses a Kimball star schema to support the four analytical questions about international football match results.

**Fact Table: fact_matches**

Grain: One row represents one unique international football match.

The fact table stores goals scored, total goals, match count, and win/draw indicators.

**Dimension Tables:**

1. dim_date: Match date, year, and decade.
2. dim_team: National team names, used for both home and away teams.
3. dim_tournament: Tournament or competition.
4. dim_venue: Match location and neutral-venue status.

The fact table connects directly to these four dimensions. The team dimension is used twice, once for the home team and once for the away team.

### Gold Star Schema creation

In [0]:
%sql
-- Create Gold Star Schema

-- Date Dimension
CREATE TABLE IF NOT EXISTS gold.dim_date (
    date DATE,
    year INT,
    decade INT
)
USING DELTA;


-- Team Dimension
CREATE TABLE IF NOT EXISTS gold.dim_team (
    team_name STRING
)
USING DELTA;


-- Tournament Dimension
CREATE TABLE IF NOT EXISTS gold.dim_tournament (
    tournament STRING
)
USING DELTA;


-- Venue Dimension
CREATE TABLE IF NOT EXISTS gold.dim_venue (
    venue_id STRING,
    city STRING,
    country STRING,
    neutral BOOLEAN
)
USING DELTA;


-- Central Fact Table
CREATE TABLE IF NOT EXISTS gold.fact_matches (
    match_id STRING,
    date DATE,
    home_team STRING,
    away_team STRING,
    tournament STRING,
    venue_id STRING,
    home_score INT,
    away_score INT,
    total_goals INT,
    home_win INT,
    away_win INT,
    draw INT,
    match_count INT,
    _load_date DATE
)
USING DELTA;

In [0]:
%sql
-- Verify Gold Tables

SHOW TABLES IN gold;

database,tableName,isTemporary
gold,dim_date,false
gold,dim_team,false
gold,dim_tournament,false
gold,dim_venue,false
gold,fact_matches,false
,stg_former_names_cleaned,true
,stg_former_names_rejected,true
,stg_goalscorers_cleaned,true
,stg_goalscorers_rejected,true
,stg_match_lookup,true


### Gold Dimension loading

In [0]:
%sql
-- Populate Gold Dimensions

-- 1. Date Dimension
MERGE INTO gold.dim_date AS t
USING (
    SELECT DISTINCT
        date,
        YEAR(date) AS year,
        CAST(FLOOR(YEAR(date) / 10) * 10 AS INT) AS decade
    FROM silver.results
    WHERE _load_date = TO_DATE(:run_date)
) AS s
ON t.date = s.date
WHEN NOT MATCHED THEN
INSERT (date, year, decade)
VALUES (s.date, s.year, s.decade);


-- 2) Team Dimension
MERGE INTO gold.dim_team AS t
USING (
    SELECT home_team AS team_name
    FROM silver.results
    WHERE _load_date = TO_DATE(:run_date)

    UNION

    SELECT away_team AS team_name
    FROM silver.results
    WHERE _load_date = TO_DATE(:run_date)
) AS s
ON t.team_name = s.team_name
WHEN NOT MATCHED THEN
INSERT (team_name)
VALUES (s.team_name);


-- 3) Tournament Dimension
MERGE INTO gold.dim_tournament AS t
USING (
    SELECT DISTINCT tournament
    FROM silver.results
    WHERE _load_date = TO_DATE(:run_date)
      AND tournament IS NOT NULL
) AS s
ON t.tournament = s.tournament
WHEN NOT MATCHED THEN
INSERT (tournament)
VALUES (s.tournament);


-- 4.) Venue Dimension
MERGE INTO gold.dim_venue AS t
USING (
    SELECT DISTINCT
        SHA2(CONCAT_WS('|',
            COALESCE(city, 'UNKNOWN'),
            COALESCE(country, 'UNKNOWN'),
            CAST(neutral AS STRING)
        ), 256) AS venue_id,
        city,
        country,
        neutral
    FROM silver.results
    WHERE _load_date = TO_DATE(:run_date)
) AS s
ON t.venue_id = s.venue_id
WHEN NOT MATCHED THEN
INSERT (venue_id, city, country, neutral)
VALUES (s.venue_id, s.city, s.country, s.neutral);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- Verify Gold Dimension Counts

SELECT 'dim_date' AS table_name, COUNT(*) AS total_rows
FROM gold.dim_date

UNION ALL

SELECT 'dim_team', COUNT(*)
FROM gold.dim_team

UNION ALL

SELECT 'dim_tournament', COUNT(*)
FROM gold.dim_tournament

UNION ALL

SELECT 'dim_venue', COUNT(*)
FROM gold.dim_venue

UNION ALL

SELECT 'fact_matches', COUNT(*)
FROM gold.fact_matches;

table_name,total_rows
dim_date,11646
dim_team,259
dim_tournament,142
dim_venue,1715
fact_matches,0


### Gold fact table loading

In [0]:
%sql
-- Populate Gold Fact Matches

MERGE INTO gold.fact_matches AS t

USING (
    SELECT
        r.match_id,
        r.date,
        r.home_team,
        r.away_team,
        r.tournament,

        SHA2(CONCAT_WS('|',
            COALESCE(r.city, 'UNKNOWN'),
            COALESCE(r.country, 'UNKNOWN'),
            CAST(r.neutral AS STRING)
        ), 256) AS venue_id,

        r.home_score,
        r.away_score,

        -- Total goals in each match
        r.home_score + r.away_score AS total_goals,

        -- Match outcome indicators
        CASE
            WHEN r.home_score > r.away_score THEN 1
            ELSE 0
        END AS home_win,

        CASE
            WHEN r.away_score > r.home_score THEN 1
            ELSE 0
        END AS away_win,

        CASE
            WHEN r.home_score = r.away_score THEN 1
            ELSE 0
        END AS draw,

        1 AS match_count,

        r._load_date

    FROM silver.results r

    WHERE r._load_date = TO_DATE(:run_date)

) AS s

ON t.match_id = s.match_id

WHEN MATCHED THEN UPDATE SET
    date = s.date,
    home_team = s.home_team,
    away_team = s.away_team,
    tournament = s.tournament,
    venue_id = s.venue_id,
    home_score = s.home_score,
    away_score = s.away_score,
    total_goals = s.total_goals,
    home_win = s.home_win,
    away_win = s.away_win,
    draw = s.draw,
    match_count = s.match_count,
    _load_date = s._load_date

WHEN NOT MATCHED THEN INSERT (
    match_id,
    date,
    home_team,
    away_team,
    tournament,
    venue_id,
    home_score,
    away_score,
    total_goals,
    home_win,
    away_win,
    draw,
    match_count,
    _load_date
)
VALUES (
    s.match_id,
    s.date,
    s.home_team,
    s.away_team,
    s.tournament,
    s.venue_id,
    s.home_score,
    s.away_score,
    s.total_goals,
    s.home_win,
    s.away_win,
    s.draw,
    s.match_count,
    s._load_date
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
11513,11513,0,0


In [0]:
%sql
-- Verify Gold Fact Matches

SELECT
    COUNT(*) AS total_matches,
    COUNT(DISTINCT match_id) AS unique_match_ids,
    SUM(match_count) AS total_match_count,
    SUM(home_win) AS home_wins,
    SUM(away_win) AS away_wins,
    SUM(draw) AS draws,
    SUM(total_goals) AS goals_scored

FROM gold.fact_matches

WHERE _load_date = TO_DATE(:run_date);

total_matches,unique_match_ids,total_match_count,home_wins,away_wins,draws,goals_scored
24066,24066,24066,12015,6721,5330,75403


In [0]:
%sql
-- Check Fact Matches Dimension Relationships

SELECT
    COUNT(*) AS missing_dimension_links

FROM gold.fact_matches f

LEFT JOIN gold.dim_date d
    ON f.date = d.date

LEFT JOIN gold.dim_team h
    ON f.home_team = h.team_name

LEFT JOIN gold.dim_team a
    ON f.away_team = a.team_name

LEFT JOIN gold.dim_tournament t
    ON f.tournament = t.tournament

LEFT JOIN gold.dim_venue v
    ON f.venue_id = v.venue_id

WHERE f._load_date = TO_DATE(:run_date)
AND (
    d.date IS NULL
    OR h.team_name IS NULL
    OR a.team_name IS NULL
    OR t.tournament IS NULL
    OR v.venue_id IS NULL
);

missing_dimension_links
0


## 7. Incremental loading - Batch 2

### Gold Fact Table — Match Results

The fact_matches table was populated using validated Silver match records.

The fact table grain is one row per unique international football match.

The numeric measures include total goals, match count, home-win indicator, away-win indicator, and draw indicator.

The fact table references the Date, Team, Tournament, and Venue dimensions.

MERGE INTO supports incremental loading by updating existing matches and inserting new ones without creating duplicate match identifiers.

In [0]:
%sql
-- Verify Batch 2 Bronze Loading

SELECT
    'results.csv' AS source_file,
    COUNT(*) AS total_rows
FROM bronze.results
WHERE _load_date = DATE '2026-10-08'

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT(*)
FROM bronze.goalscorers
WHERE _load_date = DATE '2026-10-08'

UNION ALL

SELECT
    'shootouts.csv',
    COUNT(*)
FROM bronze.shootouts
WHERE _load_date = DATE '2026-10-08';

source_file,total_rows
results.csv,14366
goalscorers.csv,14830
shootouts.csv,203


In [0]:
%sql
-- Record Batch 2 Ingestion

DELETE FROM bronze.load_audit
WHERE load_date = TO_DATE(:run_date);


-- Results Audit
INSERT INTO bronze.load_audit
SELECT
    TO_DATE(:run_date),
    'results.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*)
     FROM bronze.results
     WHERE _load_date = TO_DATE(:run_date)),
    CURRENT_TIMESTAMP()
FROM read_files(
    CONCAT(:volume_path, '/', :run_date, '/results.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);


-- Goalscorers Audit
INSERT INTO bronze.load_audit
SELECT
    TO_DATE(:run_date),
    'goalscorers.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*)
     FROM bronze.goalscorers
     WHERE _load_date = TO_DATE(:run_date)),
    CURRENT_TIMESTAMP()
FROM read_files(
    CONCAT(:volume_path, '/', :run_date, '/goalscorers.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);


-- Shootouts Audit
INSERT INTO bronze.load_audit
SELECT
    TO_DATE(:run_date),
    'shootouts.csv',
    :run_date,
    COUNT(*),
    (SELECT COUNT(*)
     FROM bronze.shootouts
     WHERE _load_date = TO_DATE(:run_date)),
    CURRENT_TIMESTAMP()
FROM read_files(
    CONCAT(:volume_path, '/', :run_date, '/shootouts.csv'),
    format => 'csv',
    header => true,
    inferSchema => false
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- Verify Batch Audit Records

SELECT
    load_date,
    source_name,
    input_rows,
    bronze_rows,
    input_rows - bronze_rows AS row_difference

FROM bronze.load_audit

WHERE load_date IN (
    DATE '2026-10-07',
    DATE '2026-10-08'
)

ORDER BY load_date, source_name;

load_date,source_name,input_rows,bronze_rows,row_difference
2026-10-07,former_names.csv,36,36,0
2026-10-07,goalscorers.csv,19202,19202,0
2026-10-07,results.csv,24066,24066,0
2026-10-07,shootouts.csv,268,268,0
2026-10-08,goalscorers.csv,14830,14830,0
2026-10-08,results.csv,14366,14366,0
2026-10-08,shootouts.csv,203,203,0


In [0]:
%sql
-- Verify Batch 2 Cleaning

SELECT
    'results.csv' AS source_file,
    COUNT(*) AS total_rows,
    COUNT_IF(rejection_reason IS NULL) AS valid_rows,
    COUNT_IF(rejection_reason IS NOT NULL) AS invalid_rows
FROM stg_results_cleaned

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_goalscorers_cleaned

UNION ALL

SELECT
    'shootouts.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_shootouts_cleaned;

source_file,total_rows,valid_rows,invalid_rows
results.csv,14366,14366,0
goalscorers.csv,14830,14830,0
shootouts.csv,203,202,1


In [0]:
%sql
-- Inspect rejected shootout records

SELECT
    date,
    home_team,
    away_team,
    winner,
    rejection_reason
FROM stg_shootouts_cleaned
WHERE rejection_reason IS NOT NULL;

date,home_team,away_team,winner,rejection_reason
2011-06-29,Saare County,Åland Islands,Åland Islands,No unique matching result


In [0]:
%sql
-- Verify Batch 2 Silver and Quarantine

SELECT
    'results.csv' AS source_file,
    (SELECT COUNT(*) FROM bronze.results
     WHERE _load_date = TO_DATE(:run_date)) AS bronze_rows,
    (SELECT COUNT(*) FROM silver.results
     WHERE _load_date = TO_DATE(:run_date)) AS silver_rows,
    (SELECT COUNT(*) FROM quarantine.results
     WHERE _load_date = TO_DATE(:run_date)) AS quarantine_rows

UNION ALL

SELECT
    'goalscorers.csv',
    (SELECT COUNT(*) FROM bronze.goalscorers
     WHERE _load_date = TO_DATE(:run_date)),
    (SELECT COUNT(*) FROM silver.goalscorers
     WHERE _load_date = TO_DATE(:run_date)),
    (SELECT COUNT(*) FROM quarantine.goalscorers
     WHERE _load_date = TO_DATE(:run_date))

UNION ALL

SELECT
    'shootouts.csv',
    (SELECT COUNT(*) FROM bronze.shootouts
     WHERE _load_date = TO_DATE(:run_date)),
    (SELECT COUNT(*) FROM silver.shootouts
     WHERE _load_date = TO_DATE(:run_date)),
    (SELECT COUNT(*) FROM quarantine.shootouts
     WHERE _load_date = TO_DATE(:run_date));

source_file,bronze_rows,silver_rows,quarantine_rows
results.csv,14366,14366,0
goalscorers.csv,14830,14830,0
shootouts.csv,203,202,1


In [0]:
%sql
-- Inspect Batch 2 Quarantine

SELECT
    date,
    home_team,
    away_team,
    winner,
    rejection_reason

FROM quarantine.shootouts

WHERE _load_date = TO_DATE(:run_date);

date,home_team,away_team,winner,rejection_reason
2011-06-29,Saare County,Åland Islands,Åland Islands,No unique matching result


In [0]:
%sql
-- Verify Batch 1 and Batch 2 Are Preserved

SELECT
    _load_date,
    COUNT(*) AS total_matches
FROM silver.results
GROUP BY _load_date
ORDER BY _load_date;

_load_date,total_matches
2026-10-07,24066
2026-10-08,14366


In [0]:
%sql
-- Verify current batch date

SELECT :run_date AS load_date;

load_date
2026-10-08


In [0]:
%sql
-- Verify Gold Dimension Counts

SELECT 'dim_date' AS table_name, COUNT(*) AS total_rows
FROM gold.dim_date

UNION ALL

SELECT 'dim_team', COUNT(*)
FROM gold.dim_team

UNION ALL

SELECT 'dim_tournament', COUNT(*)
FROM gold.dim_tournament

UNION ALL

SELECT 'dim_venue', COUNT(*)
FROM gold.dim_venue

UNION ALL

SELECT 'fact_matches', COUNT(*)
FROM gold.fact_matches;

table_name,total_rows
dim_date,14906
dim_team,307
dim_tournament,175
dim_venue,2617
fact_matches,24066


In [0]:
%sql
-- Verify Incremental Gold Fact Matches

SELECT
    _load_date,
    COUNT(*) AS total_matches,
    COUNT(DISTINCT match_id) AS unique_match_ids,
    SUM(match_count) AS total_match_count

FROM gold.fact_matches

GROUP BY _load_date

ORDER BY _load_date;

_load_date,total_matches,unique_match_ids,total_match_count
2026-10-07,24066,24066,24066
2026-10-08,14366,14366,14366


## 8. Incremental Loading Batch 3

In [0]:
%sql
-- Verify Batch 3 Bronze Loading

SELECT
    'results.csv' AS source_file,
    COUNT(*) AS total_rows
FROM bronze.results
WHERE _load_date = DATE '2026-10-09'

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT(*)
FROM bronze.goalscorers
WHERE _load_date = DATE '2026-10-09'

UNION ALL

SELECT
    'shootouts.csv',
    COUNT(*)
FROM bronze.shootouts
WHERE _load_date = DATE '2026-10-09';

source_file,total_rows
results.csv,11513
goalscorers.csv,13882
shootouts.csv,229


In [0]:
%sql
-- Verify All Three Batch Audit Records

SELECT
    load_date,
    source_name,
    input_rows,
    bronze_rows,
    input_rows - bronze_rows AS row_difference

FROM bronze.load_audit

WHERE load_date IN (
    DATE '2026-10-07',
    DATE '2026-10-08',
    DATE '2026-10-09'
)

ORDER BY load_date, source_name;

load_date,source_name,input_rows,bronze_rows,row_difference
2026-10-07,former_names.csv,36,36,0
2026-10-07,goalscorers.csv,19202,19202,0
2026-10-07,results.csv,24066,24066,0
2026-10-07,shootouts.csv,268,268,0
2026-10-08,goalscorers.csv,14830,14830,0
2026-10-08,results.csv,14366,14366,0
2026-10-08,shootouts.csv,203,203,0
2026-10-09,goalscorers.csv,13882,13882,0
2026-10-09,results.csv,11513,11513,0
2026-10-09,shootouts.csv,229,229,0


In [0]:
%sql
-- Verify Batch 3 date

SELECT :run_date AS load_date;

load_date
2026-10-09


In [0]:
%sql
-- Verify Batch 3 Cleaning

SELECT
    'results.csv' AS source_file,
    COUNT(*) AS total_rows,
    COUNT_IF(rejection_reason IS NULL) AS valid_rows,
    COUNT_IF(rejection_reason IS NOT NULL) AS invalid_rows
FROM stg_results_cleaned

UNION ALL

SELECT
    'goalscorers.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_goalscorers_cleaned

UNION ALL

SELECT
    'shootouts.csv',
    COUNT(*),
    COUNT_IF(rejection_reason IS NULL),
    COUNT_IF(rejection_reason IS NOT NULL)
FROM stg_shootouts_cleaned;

source_file,total_rows,valid_rows,invalid_rows
results.csv,11513,11513,0
goalscorers.csv,13882,13882,0
shootouts.csv,229,229,0


### Batch 3 — Silver Cleaning and Validation

The third historical batch was processed using the same SQL cleaning and validation logic applied to the previous batches.

The process checks dates, team names, scores, goal indicators, goal minutes, and match relationships.

The batch-specific temporary staging views were recreated using the selected load date.

Records passing validation are prepared for Silver, while records failing validation will be directed to Quarantine.

This demonstrates a repeatable incremental cleaning process.

In [0]:
%sql
--Verify Batch 3 Silver and Quarantine

SELECT
    'results.csv' AS source_file,
    (SELECT COUNT(*) FROM silver.results
     WHERE _load_date = DATE '2026-10-09') AS silver_rows,
    (SELECT COUNT(*) FROM quarantine.results
     WHERE _load_date = DATE '2026-10-09') AS quarantine_rows

UNION ALL

SELECT
    'goalscorers.csv',
    (SELECT COUNT(*) FROM silver.goalscorers
     WHERE _load_date = DATE '2026-10-09'),
    (SELECT COUNT(*) FROM quarantine.goalscorers
     WHERE _load_date = DATE '2026-10-09')

UNION ALL

SELECT
    'shootouts.csv',
    (SELECT COUNT(*) FROM silver.shootouts
     WHERE _load_date = DATE '2026-10-09'),
    (SELECT COUNT(*) FROM quarantine.shootouts
     WHERE _load_date = DATE '2026-10-09');

source_file,silver_rows,quarantine_rows
results.csv,11513,0
goalscorers.csv,13882,0
shootouts.csv,229,0


In [0]:
%sql
-- Verify Batch 3 Gold Dimensions

SELECT 'dim_date' AS table_name, COUNT(*) AS total_rows
FROM gold.dim_date

UNION ALL

SELECT 'dim_team', COUNT(*)
FROM gold.dim_team

UNION ALL

SELECT 'dim_tournament', COUNT(*)
FROM gold.dim_tournament

UNION ALL

SELECT 'dim_venue', COUNT(*)
FROM gold.dim_venue

UNION ALL

SELECT 'fact_matches', COUNT(*)
FROM gold.fact_matches;

table_name,total_rows
dim_date,16535
dim_team,337
dim_tournament,225
dim_venue,3166
fact_matches,38432


In [0]:
%sql
-- Verify All Gold Match Batches

SELECT
    _load_date,
    COUNT(*) AS total_matches,
    COUNT(DISTINCT match_id) AS unique_match_ids,
    SUM(match_count) AS total_match_count

FROM gold.fact_matches

GROUP BY _load_date

ORDER BY _load_date;

_load_date,total_matches,unique_match_ids,total_match_count
2026-10-07,24066,24066,24066
2026-10-08,14366,14366,14366
2026-10-09,11513,11513,11513


In [0]:
%sql
-- Verify the Lakehouse tables still contain data

SELECT
    'Bronze Results' AS table_name,
    COUNT(*) AS total_rows
FROM bronze.results

UNION ALL

SELECT
    'Silver Results',
    COUNT(*)
FROM silver.results

UNION ALL

SELECT
    'Gold Date Dimension',
    COUNT(*)
FROM gold.dim_date;

table_name,total_rows
Bronze Results,49945
Silver Results,49945
Gold Date Dimension,16535


## 9. Final Data Quality Checks

In [0]:
%sql
-- Final Pipeline Row Counts

SELECT
    'results.csv' AS source_file,
    (SELECT COUNT(*) FROM bronze.results) AS bronze_rows,
    (SELECT COUNT(*) FROM silver.results) AS silver_rows,
    (SELECT COUNT(*) FROM quarantine.results) AS quarantine_rows

UNION ALL

SELECT
    'goalscorers.csv',
    (SELECT COUNT(*) FROM bronze.goalscorers),
    (SELECT COUNT(*) FROM silver.goalscorers),
    (SELECT COUNT(*) FROM quarantine.goalscorers)

UNION ALL

SELECT
    'shootouts.csv',
    (SELECT COUNT(*) FROM bronze.shootouts),
    (SELECT COUNT(*) FROM silver.shootouts),
    (SELECT COUNT(*) FROM quarantine.shootouts)

UNION ALL

SELECT
    'former_names.csv',
    (SELECT COUNT(*) FROM bronze.former_names),
    (SELECT COUNT(*) FROM silver.former_names),
    (SELECT COUNT(*) FROM quarantine.former_names);

source_file,bronze_rows,silver_rows,quarantine_rows
results.csv,49945,49945,0
goalscorers.csv,47914,47914,0
shootouts.csv,700,699,1
former_names.csv,36,36,0


In [0]:
%sql
-- Final Gold Match Quality Checks

SELECT
    COUNT(*) AS total_matches,

    COUNT(DISTINCT match_id) AS unique_match_ids,

    COUNT_IF(match_id IS NULL) AS null_match_ids,

    SUM(match_count) AS total_match_count,

    SUM(home_win + away_win + draw) AS classified_matches

FROM gold.fact_matches;

total_matches,unique_match_ids,null_match_ids,total_match_count,classified_matches
49945,49945,0,49945,49945


In [0]:
%sql
-- Check Gold Dimension Relationships

SELECT
    COUNT(*) AS missing_dimension_links

FROM gold.fact_matches f

LEFT JOIN gold.dim_date d
    ON f.date = d.date

LEFT JOIN gold.dim_team h
    ON f.home_team = h.team_name

LEFT JOIN gold.dim_team a
    ON f.away_team = a.team_name

LEFT JOIN gold.dim_tournament t
    ON f.tournament = t.tournament

LEFT JOIN gold.dim_venue v
    ON f.venue_id = v.venue_id

WHERE
    d.date IS NULL
    OR h.team_name IS NULL
    OR a.team_name IS NULL
    OR t.tournament IS NULL
    OR v.venue_id IS NULL;

missing_dimension_links
0


## 10. Gold Star Schema Diagram


![match_performance_star_schema_1791999514321.png](./match_performance_star_schema_1791999514321.png "match_performance_star_schema_1791999514321.png")

The Gold layer follows a Kimball star-schema design.

The central fact table is fact_matches, with one row representing one international football match.

Four dimension tables are connected to the fact table: dim_date, dim_team, dim_tournament, and dim_venue.

The dim_team table is used for both home and away teams.

Primary-key and foreign-key relationships are shown in the diagram. These are logical relationships checked using SQL.




## 11. Analytical queestions and visualizations

###Question 1: Average Goals Across Decades

How has the average number of goals per international match changed across decades?

This query uses the Gold fact_matches table and dim_date dimension to calculate the number of matches, total goals, and average goals per match in each decade.

In [0]:
%sql
-- Question 1 - Average Goals Across Decades

SELECT
    d.decade,
    COUNT(*) AS total_matches,
    SUM(f.total_goals) AS total_goals,
    ROUND(AVG(f.total_goals), 2) AS avg_goals_per_match

FROM gold.fact_matches f

JOIN gold.dim_date d
    ON f.date = d.date

GROUP BY d.decade

ORDER BY d.decade;

decade,total_matches,total_goals,avg_goals_per_match
1870,13,59,4.54
1880,55,307,5.58
1890,59,304,5.15
1900,138,574,4.16
1910,330,1392,4.22
1920,831,3228,3.88
1930,1079,4660,4.32
1940,833,3619,4.34
1950,1657,6629,4.0
1960,2971,10336,3.48


Databricks visualization. Run in Databricks to view.

### Question 1 — Interpretation

The results show that the average number of goals per international match generally decreased over time. In the 1870s, matches averaged 4.54 goals, compared with 2.73 goals in the 2020s. The 1880s had the highest average, with 5.58 goals per match.

Although the overall trend is downward, some decades experienced increases. Early decades also contained fewer recorded matches, and the 2020s are incomplete. These limitations should be considered when comparing the results.

### Question 2: Home Wins by Venue Type

How does the percentage of home-team victories compare between neutral and non-neutral venues?

This query uses the Gold fact_matches table and dim_venue dimension to compare home-team victory percentages across the two venue types.

In [0]:
%sql
--Question 2 - Home Wins by Venue Type

SELECT
    CASE
        WHEN v.neutral = TRUE THEN 'Neutral'
        ELSE 'Non-neutral'
    END AS venue_type,

    COUNT(*) AS total_matches,

    SUM(f.home_win) AS home_wins,

    ROUND(
        100.0 * SUM(f.home_win) / COUNT(*), 2
    ) AS home_win_percentage

FROM gold.fact_matches f

JOIN gold.dim_venue v
    ON f.venue_id = v.venue_id

GROUP BY v.neutral

ORDER BY v.neutral;

venue_type,total_matches,home_wins,home_win_percentage
Non-neutral,36669,18603,50.73
Neutral,13276,5870,44.22


Databricks visualization. Run in Databricks to view.

### Question 2 — Interpretation

The results show that designated home teams won 50.73% of matches played at non-neutral venues, compared with 44.22% at neutral venues. This represents a difference of 6.51 percentage points.

This comparison suggests that home teams have historically won a higher proportion of matches at non-neutral venues in this dataset. However, the results describe an association rather than proving that playing at home causes a team to win.

### Question 3: Top 10 Non-Friendly Tournaments

Which 10 non-friendly tournaments have hosted the largest number of international matches since 2000?

This query uses the Gold fact_matches table, dim_tournament, and dim_date to rank tournaments according to their number of recorded matches.

Friendly matches are excluded, and only matches played from 2000 onward are included.

In [0]:
%sql
-- Question 3 - Top 10 Non-Friendly Tournaments

SELECT
    t.tournament,
    COUNT(*) AS total_matches

FROM gold.fact_matches f

JOIN gold.dim_tournament t
    ON f.tournament = t.tournament

JOIN gold.dim_date d
    ON f.date = d.date

WHERE d.year >= 2000
  AND LOWER(TRIM(t.tournament)) <> 'friendly'

GROUP BY t.tournament

ORDER BY total_matches DESC, t.tournament ASC

LIMIT 10;

tournament,total_matches
FIFA World Cup qualification,5840
UEFA Euro qualification,1532
African Cup of Nations qualification,1476
UEFA Nations League,762
AFC Asian Cup qualification,530
African Cup of Nations,525
CONCACAF Nations League,495
FIFA World Cup,488
Gold Cup,359
CECAFA Cup,341


Databricks visualization. Run in Databricks to view.

### Question 3 — Interpretation

The results show that FIFA World Cup qualification had the highest number of recorded international matches since 2000, with 5,840 matches. UEFA Euro qualification ranked second with 1,532 matches, followed by African Cup of Nations qualification with 1,476 matches.

This indicates that qualification competitions contribute a large proportion of the recorded non-friendly international matches. However, the rankings reflect total match counts rather than the competitiveness or importance of each tournament.

###Question 4: Top 10 National Teams by Victories

Which 10 national teams have recorded the most match victories since 2000, based on final match scores?

This query uses the Gold fact_matches table, dim_date, and dim_team to identify and rank national teams by their number of recorded victories since 2000.

Both home and away victories are included. Draws are excluded, and results are determined from the recorded match scores rather than penalty shootout outcomes.

In [0]:
%sql
-- Question 4 - Top 10 Teams by Victories

WITH match_winners AS (

    SELECT
        CASE
            WHEN f.home_win = 1 THEN f.home_team
            WHEN f.away_win = 1 THEN f.away_team
            ELSE NULL
        END AS winning_team

    FROM gold.fact_matches f

    JOIN gold.dim_date d
        ON f.date = d.date

    WHERE d.year >= 2000
)

SELECT
    t.team_name,
    COUNT(*) AS total_wins

FROM match_winners w

JOIN gold.dim_team t
    ON w.winning_team = t.team_name

GROUP BY t.team_name

ORDER BY total_wins DESC, t.team_name ASC

LIMIT 10;

team_name,total_wins
Mexico,261
Spain,246
Brazil,242
United States,237
Japan,229
Argentina,226
France,226
Iran,214
Germany,212
Egypt,205


Databricks visualization. Run in Databricks to view.

### Question 4 — Interpretation

The results show that Mexico recorded the highest number of international match victories since 2000, with 261 wins. Spain ranked second with 246 victories, followed by Brazil with 242, the United States with 237, and Japan with 229.

These results highlight the national teams with the greatest number of recorded victories during the selected period. However, the ranking is based on total wins rather than win percentages. Teams may have played different numbers of matches, so the results do not necessarily indicate which team had the highest success rate.

## 12. Final Reflection

My main design decision was to use a match-level fact table rather than copying the entire results dataset into the Gold layer. Each row in fact_matches represents one international match, and date, team, tournament, and venue dimensions provide the context needed for the four questions. The same team dimension plays two roles, for home and away teams. This kept the model simple and made it possible to calculate goal averages, home-win percentages, tournament match rankings, and team victories with SQL joins and aggregations.

One data-quality challenge was that a date and pair of teams did not always uniquely identify a match. We found two Tahiti–New Caledonia records on the same date with different scores. Automatically removing one would have discarded a possibly valid match, so I used a match identifier based on more match attributes. The Silver pipeline also checked dates, scores, team names, and match relationships, while keeping rejected records separately in Quarantine. One unmatched 2011 shootout was quarantined instead of being silently ignored. The Bronze-to-Silver row counts and key checks helped confirm that trusted match data reached Gold without unexpected losses.

A limitation is that the dataset's recorded home-team designation does not necessarily mean the match took place in that team's country, and the analysis cannot establish a causal home advantage. If updated regularly, I would improve the pipeline by tracking source changes and stable match identifiers across batches, so corrected historical results can be updated reliably without duplicates.